In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, f1_score


In [2]:
# load the scaled dataset
df_logreg = pd.read_csv("../data/processed/features_2023_scaled.csv")

In [3]:
print(f"Shape of the dataset: {df_logreg.shape}\n\n")
print("Races available: ",  df_logreg["RaceName"].unique())

Shape of the dataset: (11089, 26)


Races available:  ['Abu Dhabi' 'Australia' 'Bahrain' 'Hungary' 'Monaco' 'Monza'
 'Saudi Arabia' 'Silverstone' 'Singapore' 'Spain']


```
Why we split by race, not randomly 

train, test = train_test_split(df_logreg, test_size = 0.2, random_state 42)

a random split picks rows completely randomly, regardless of which race they came from. This means we'd likely end up with laps from the SAME race in BOTH training set and test set.


DATA LEAKAGE!
laps from the same race are not independent of each other.They share context, same track temperature, same safety care periods, same overall race strategy patterns, similar tyre degradation curves for that specific circuit.

If the model sees SOME laps from Bahrain during training, then gets tested on OTHER laps from the SAME bahrain race, it's not really being tested on 'unseen' data. It is already partially learned that specific race's conditions and patterns. This makes the test result look artificially better than they actually are

This is called a data leakage, information leaking between train and test sets that shouldn't be there in a fair evaluation.
```


 ----

```

Choosing test races:

Abu Dhabi     → modern, medium-speed circuit, lots of overtaking zones
Australia     → semi-street circuit, moderate degradation
Bahrain       → high degradation, desert conditions
Hungary       → very high degradation, low overtaking
Monaco        → street circuit, lowest degradation, no real pit strategy battles
Monza         → very low degradation, high speed
Saudi Arabia  → street circuit, high speed
Silverstone   → classic circuit, high speed corners
Singapore     → street circuit, night race, high safety car probability
Spain         → medium degradation, technical circuit



Singapore → street circuit, high safety car chance, different strategy dynamics
Monza     → very different track type, low degradation, high speed

Together they represent genuinely DIFFERENT racing conditions
from most of our training races, which makes for a fairer,
more challenging test of whether our model truly generalizes
```

 -----

In [8]:
# Split races 
test_races = ["Singapore", "Monza"]

# give me everything NOT in this list
train_df = df_logreg[~df_logreg["RaceName"].isin(test_races)]
test_df = df_logreg[df_logreg["RaceName"].isin(test_races)]

print(f"\n\n Train shape: {train_df.shape}")
print(f"Test shape:       {test_df.shape}\n\n")

print(f"Train races: {train_df['RaceName'].unique()}")
print(f"Test races:  {test_df['RaceName'].unique()}")


                                        



 Train shape: (9079, 26)
Test shape:       (2010, 26)


Train races: ['Abu Dhabi' 'Australia' 'Bahrain' 'Hungary' 'Monaco' 'Saudi Arabia'
 'Silverstone' 'Spain']
Test races:  ['Monza' 'Singapore']


In [9]:
# Seperate Features (X) from Target (y)

# Pitted = target, we are predicting this
# Driver, RaceName, Compound = identity/text columns, not used as model inputs

exclude_cols = ["Pitted", "Driver", "RaceName", "Compound",
               "IsAccurate", "FastF1Generated", "IsPersonalBest"]

feature_cols = [col for col in train_df.columns if col  not in exclude_cols]

X_train = train_df[feature_cols]
y_train = train_df["Pitted"]

X_test = test_df[feature_cols]
y_test = test_df["Pitted"]


print(f"Feature columns: {feature_cols}")
print(f"\n X_train shape: {X_train.shape}")
print(f"\n y_train shape: {y_train.shape}")

Feature columns: ['Stint', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'TyreLife', 'FreshTyre', 'Position', 'CompoundEncoded', 'LapTimeDelta', 'LapTimeRolling', 'DegradationFromStintStart', 'TotalLaps', 'RacePctComplete', 'LapsRemaining', 'IsLateRace']

 X_train shape: (9079, 19)

 y_train shape: (9079,)


In [10]:
# Training the model

# class_weight = 'balanced', handles our severe class imbalance
# 33 to 1 ratio, without this model would just predict 'never pit'


model = LogisticRegression(class_weight = 'balanced', max_iter = 1000, random_state = 42)


model.fit(X_train, y_train)
print("Model trained")


Model trained


In [16]:
# Predictions

# get predictions on the unseen test races
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("Logistic Regression Performance (Singapore + Monza, unseen):\n")
print(classification_report(y_test, y_pred))

auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC: {auc:.4f}")


Logistic Regression Performance (Singapore + Monza, unseen):

              precision    recall  f1-score   support

           0       0.99      0.50      0.66      1959
           1       0.04      0.80      0.08        51

    accuracy                           0.51      2010
   macro avg       0.51      0.65      0.37      2010
weighted avg       0.97      0.51      0.65      2010

ROC-AUC: 0.7044


``` ROC - AUC: 0.60 (baseline) --> 0.8362 (logistic regrssion)
this is an imporovement in the model's ability to RANK pit-laps as more like than non pit-laps.

Class 1 (pitted):
precision: 0.04 --> still terrible, even worse than baseline!
recall: 1.00 ------> catches evey single real pit stop

a model that catches 100% of pit stops while having only 4% precision means it's basically just predicting PIT for almost everything
```

In [ ]:
print("Predicted pit count:", (y_pred == 1).sum())
print("Actual pit count:", (y_test ==1).sum())

```

Reason: class_weight = "balanced" went too far
it essentially shifted its decision boundary so aggresively toward catching positive that it now fires "pit" very liberaly.

ROC-AUC (0.83)        → measures: can the model correctly RANK/SCORE 
                          pit-laps as more likely than non-pit-laps?
                          → YES, this is genuinely good

Precision/Recall (0.5 threshold) → measures: using a simple "yes/no at 50% cutoff",
                          how good are the actual decisions?
                          → NO, the 0.5 cutoff is badly calibrated here

```

In [ ]:
# Adjusting the decision threshold 
# Instead of using the default 0.5 cutoff, finding something better.

precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba)

# Plot precision and recall against different thresholds
plt.figure(figsize = (10,5))
plt.plot(thresholds, precisions[:-1], label = "Precision")
plt.plot(thresholds, recalls[:-1], label = "Recall")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Preision-Recall tradeoff at Different Thresholds")
plt.legend()
plt.grid(True, alpha = 0.3)
plt.show()

```
Recall (orange line)  → stays near 1.0 almost the entire way,
                         only drops sharply right at the very end (threshold ~0.95+)

Precision (blue line) → stays extremely low (around 0.03-0.06) 
                         across almost the ENTIRE threshold range,
                         only spikes up at the very last moment (threshold ~0.98+)


this is a sign that the model's predicted probabilities are poorly calibrated. meaning most of its probabilitiy outputs are clustred in a narrow range and very few laps get a genuinely HIGH confidence 'This is a pit lap' score.

In a well behaved model, we'd expect to see precision and recall cross somewhere in the middle of the threshold range (like around 0.3 -0.5) giving us a sensible tradeoff point.
Here they barely move until threshold 0.95+ which tells us the model is assigning most laps fairly similar, generally low probabilities

It is not confidently seperating 'definitely pit' laps froom 'definitely not' laps across most of the range.

```

```
Why this might be happening:

1. The features don't have enough genuinely distinguishing signal at the indiviual-lap level (pit decisions depend heavily on context the model doesn't fully captyre)

2. class_weight = 'balanced' has pushed the model to be overly 'trigger happy' - predicing moderate to high probabilities for almost everything, rather than being genuinely confident only on true pit laps.

3.Logistix regression's LINEAR DECISION BOUNDARY genuinely struggles here,
out pit decisions depend on complext NON- LINEAR interactions between tyre age, compound, position, laps remaining etc. A straight line boundary just can't carve out pit-laps cleanly.

```

In [ ]:
# Finding where the curves actually  cross 
# best (for now) balance point using F1 score across thresholds 


f1_scores = []
for threshold in np.arange(0.1, 1.0, 0.01):
    y_pred_thresh = (y_pred_proba >= threshold).astype(int)
    f1 = f1_score(y_test, y_pred_thresh, zero_division=0)
    f1_scores.append(f1)

best_threshold_idx = np.argmax(f1_scores)
best_threshold = np.arange(0.1, 1.0, 0.01)[best_threshold_idx]
best_f1 = f1_scores[best_threshold_idx]

print(f"Best threshold: {best_threshold:.2f}")
print(f"Best F1 score at that threshold: {best_f1:.3f}")

```
best threshold is 0.99, it means the model has to be almost 100% certain before it's worth trusting a 'pit' prediction. and even then, the best F1 score we can squeeze out is only 0.208.
```

```
Baseline (tyre age rule):     AUC = 0.60, essentially unusable
Logistic Regression:          AUC = 0.84 (decent ranking ability)
                               but F1 = 0.21 even at best threshold
                               (struggles to make confident, precise decisions)

→ This strongly suggests the relationship between features 
  and pit decisions is NON-LINEAR, and a model that can 
  capture complex feature interactions (like XGBoost) 
  should perform meaningfully better

```

In [ ]:
y_pred_best = (y_pred_proba >= best_threshold).astype(int)
print(f"Logistic Regression with optimized threshold ({best_threshold:.2f}):\n")
print(classification_report(y_test, y_pred_best))

```

Class 0 (didn't pit)
precision: 0.99 → when model says "won't pit", it's right 99% of the time
recall:    0.91 → catches 91% of all the genuine "didn't pit" laps
f1-score:  0.94 → excellent overall balance

- This makes sense — most laps genuinely don't involve pitting, so the model is naturally good at recognizing "normal racing" laps.


Class 1 (pitted)
precision: 0.12 → when model says "pit now", it's only right 13% of the time
recall:    0.51 → catches about half (51%) of all the real pit stops
f1-score:  0.20 → still quite low overall

accuracy:     0.96  → DON'T trust this number for imbalanced data!
                       90% accuracy SOUNDS great, but remember —
                       just predicting "never pit" for everything 
                       would already give you 97% accuracy
                       (1959/2010), so 90% is actually mediocre here

macro avg:    treats both classes equally (averages 0 and 1 scores
              with EQUAL weight, regardless of how many examples 
              each class has)
              precision: (0.99+0.13)/2 = 0.56
              this shows the TRUE struggle — macro avg precision 
              of 0.56 reveals the model is genuinely only "okay"
              when you don't let the huge class 0 sample size 
              hide class 1's problems

weighted avg: weights by how many samples are in each class
              (so class 0, with 1959 samples, dominates this average)
              this is why weighted avg precision looks like 0.96 — 
              misleadingly high, because class 0 is so much bigger



It can catch about half of real pit stops (recall 0.51)
But when it DOES say "pit", it's wrong 87% of the time (precision 0.13)
```

In [ ]:
# Model coefficients 
# this tells us, what the model actually learned, which features push toward "pit",
# which push toward "stay out"

coefficients = pd.DataFrame ({
    "Feature": X_train.columns,
    "Coefficients": model.coef_[0]
}).sort_values("Coefficients", ascending = False)


print(coefficients)

```
Positive coefficient → pushes prediction TOWARDS "pit" (1)
Negative coefficient → pushes prediction TOWARDS "stay out" (0)
Bigger absolute value → stronger influence on the decision
```

## Summary — Logistic Regression Results

**Test set:** Singapore + Monza (held out, unseen races)

| Model | ROC-AUC | F1 (class 1) | Precision (class 1) | Recall (class 1) |
|---|---|---|---|---|
| Baseline (tyre age rule) | 0.60 | 0.09 | 0.05 | 0.48 |
| Logistic Regression | 0.84 | 0.21 | 0.13 | 0.51 |

**Findings:**
- Logistic Regression significantly improved ranking ability (AUC) over the baseline
- However, even after threshold tuning, precision remains low — the model 
  struggles to make confident, reliable "pit now" decisions
- This suggests the relationship between race features and pit decisions 
  is non-linear, motivating a move to a tree-based model (XGBoost) that 
  can capture complex feature interactions